In [1]:
import torch 
import json, math, random, re, time
from pathlib import Path
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

In [2]:
# get model imports
from transformers import (Qwen2VLForConditionalGeneration, AutoProcessor,
                          BitsAndBytesConfig, get_cosine_schedule_with_warmup)

In [3]:
# get LoRA imports 
from peft import LoraConfig, get_peft_model

In [4]:
# seeding for reproducibility
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

In [5]:
# define paths 
BASE        = Path("../dataset")
OUT_DIR     = Path("../outputs/finetuned_v1")
ADAPTER_DIR = Path("../adapters/lora_v1")
LOG_DIR     = Path("../logs")
for d in (OUT_DIR, ADAPTER_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)


In [17]:
# set configs

MODEL_ID     = "Qwen/Qwen2-VL-7B-Instruct"
PIXEL_BUDGET = None
EPOCHS       = 3
LR           = 1e-4
GRAD_ACCUM   = 4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
DEBUG_STEPS  = 4              # set to 4 for the memory test, then None

In [7]:
# training and validation split
with open(BASE / "sequences.json") as f:
    sequences = json.load(f)
dev_seqs = [s for s in sequences if s["split"] == "development"]

labels = {}
with open(BASE / "development/labels.jsonl") as f:
    for line in f:
        d = json.loads(line)
        labels[d["sequence_id"]] = d["pages"]

ids = [s["sequence_id"] for s in dev_seqs]
random.Random(SEED).shuffle(ids)
val_ids, train_ids = set(ids[:16]), set(ids[16:])
train_seqs = [s for s in dev_seqs if s["sequence_id"] in train_ids]
val_seqs   = [s for s in dev_seqs if s["sequence_id"] in val_ids]

with open(OUT_DIR / "split.json", "w") as f:
    json.dump({"train": sorted(train_ids), "val": sorted(val_ids)}, f, indent=1)
print(len(train_seqs), "train |", len(val_seqs), "val")

64 train | 16 val


In [8]:
# load qwen-model and add LoRA

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map="cuda", torch_dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],   # language model only
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"GPU memory after load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 8,331,745,792 || trainable%: 0.4845
GPU memory after load: 5.68 GB


In [ ]:
# prompt, resizing and dataset

PROMPT = (
    "These are 3 consecutive pages of an English manga, in order. "
    "Extract the story text of each page in reading order and say who speaks each line. "
    "Label characters character1, character2, ... consistently across all 3 pages. "
    "Use UNKNOWN if the speaker is unclear and NARRATION for narrator boxes. "
    "Exclude sound effects, titles, credits, page numbers, and text on signs or documents. "
    'Output JSON only: {"pages":[[...],[...],[...]]}, one list per page, '
    "with objects {\"speaker\":...,\"text\":...}. Use an empty list for a page with no story text."
)

def fit_image(img, budget=PIXEL_BUDGET):
    budget = PIXEL_BUDGET if budget is None else budget
    if budget is None:
        return img  
    w, h = img.size
    scale = min(1.0, (budget / (w * h)) ** 0.5)
    w2 = max(28, int(w * scale) // 28 * 28)
    h2 = max(28, int(h * scale) // 28 * 28)
    return img.resize((w2, h2), Image.LANCZOS)

def load_images(seq):
    return [fit_image(Image.open(BASE / p).convert("RGB")) for p in seq["images"]]

def make_messages(images):
    content = []
    for i, img in enumerate(images):
        content.append({"type": "text", "text": f"Page {i+1}:"})
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": PROMPT})
    return [{"role": "user", "content": content}]

def target_text(seq_id):
    return json.dumps({"pages": labels[seq_id]}, ensure_ascii=False, separators=(",", ":"))

class MangaSeqDataset(Dataset):
    def __init__(self, seqs):
        self.seqs = seqs
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, i):
        seq = self.seqs[i]
        images = load_images(seq)
        prompt_text = processor.apply_chat_template(
            make_messages(images), tokenize=False, add_generation_prompt=True)
        enc = processor(text=[prompt_text], images=images, return_tensors="pt")
        tgt = processor.tokenizer(target_text(seq["sequence_id"]) + "<|im_end|>",
                                add_special_tokens=False, return_tensors="pt").input_ids

        input_ids = torch.cat([enc.input_ids, tgt], dim=1)
        labels_t  = torch.cat([torch.full_like(enc.input_ids, -100), tgt], dim=1)

        # image-token marker: use the processor's, else rebuild it from the image token id
        if "mm_token_type_ids" in enc:
            mm = enc["mm_token_type_ids"]
        else:
            mm = (enc.input_ids == model.config.image_token_id).long()
        mm = torch.cat([mm, torch.zeros_like(tgt)], dim=1)

        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": labels_t,
            "mm_token_type_ids": mm,
            "pixel_values": enc.pixel_values,
            "image_grid_thw": enc.image_grid_thw,
        }

train_loader = DataLoader(MangaSeqDataset(train_seqs), batch_size=1, shuffle=True,
                          collate_fn=lambda b: b[0])
val_loader   = DataLoader(MangaSeqDataset(val_seqs), batch_size=1, shuffle=False,
                          collate_fn=lambda b: b[0])

In [ ]:
tok = processor.tokenizer
tgt_lens = [len(tok(target_text(s["sequence_id"]), add_special_tokens=False).input_ids)
            for s in dev_seqs]
print(f"Answer tokens per sequence: mean {sum(tgt_lens)/len(tgt_lens):.0f}, max {max(tgt_lens)}")
MAX_NEW_TOKENS = min(int(max(tgt_lens) * 1.25) + 50, 3000)
print("MAX_NEW_TOKENS for inference:", MAX_NEW_TOKENS)

item = MangaSeqDataset(train_seqs)[0]
n_img = int((item["image_grid_thw"].prod(-1) // 4).sum())
print("input_ids:", tuple(item["input_ids"].shape), "| image tokens:", n_img)
print("supervised (non -100) tokens:", int((item["labels"] != -100).sum()))

Answer tokens per sequence: mean 428, max 965
MAX_NEW_TOKENS for inference: 1256
input_ids: (1, 4947) | image tokens: 4554
supervised (non -100) tokens: 233


In [ ]:
def to_cuda(batch):
    out = {k: v.to("cuda") for k, v in batch.items()}
    out["pixel_values"] = out["pixel_values"].to(torch.bfloat16)
    return out

@torch.no_grad()
def eval_loss(loader):
    model.eval()
    tot, n = 0.0, 0
    for batch in loader:
        tot += model(**to_cuda(batch)).loss.item(); n += 1
    model.train()
    return tot / n

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, total_steps // 10), total_steps)

log_path = LOG_DIR / "finetune_v1.jsonl"
model.train()
start, micro = time.time(), 0
for epoch in range(EPOCHS):
    run_loss, run_n = 0.0, 0
    for step, batch in enumerate(train_loader):
        loss = model(**to_cuda(batch)).loss
        (loss / GRAD_ACCUM).backward()
        run_loss += loss.item(); run_n += 1; micro += 1

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        if micro % 8 == 0:
            print(f"ep {epoch+1} ex {step+1}/{len(train_loader)} "
                  f"loss {run_loss/run_n:.4f} | {time.time()-start:.0f}s | "
                  f"peak {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")

        if DEBUG_STEPS and micro >= DEBUG_STEPS:
            print(f"Debug run OK. {(time.time()-start)/micro:.1f} s/example, "
                  f"peak {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")
            raise SystemExit

    v = eval_loss(val_loader)
    rec = {"epoch": epoch + 1, "train_loss": run_loss / run_n, "val_loss": v,
           "elapsed_s": time.time() - start}
    print(rec)
    with open(log_path, "a") as f:
        f.write(json.dumps(rec) + "\n")
    model.save_pretrained(ADAPTER_DIR / f"epoch_{epoch+1}")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Debug run OK. 84.8 s/example, peak 19.2 GB


SystemExit: 

/home/rahul/byte-applied-ai-manga-task-2026/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Memory test results: 

3 pages at full resolution peaks at 19.2 GB (5.68 GB base + ~13.5 GB for forward/backward pass on 4,554 image tokens)


Decision: train per-page with previous pages’ labels passed as context text

This keeps full image resolution (best text reading quality), fits in 12 GB VRAM, trains ~3× faster, and teaches cross-page character consistency through context rather than joint attention.

In [ ]:
# new configs

MODEL_ID     = "Qwen/Qwen2-VL-7B-Instruct"
PIXEL_BUDGET = None              # None = full resolution (one page fits, we test below)
EPOCHS       = 3
LR           = 1e-4
GRAD_ACCUM   = 4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
DEBUG_STEPS  = 4                 # memory test on the 4 hardest examples; set None for real run

In [9]:
PAGE_PROMPT = (
    "This is page {n} of 3 of an English manga sequence. "
    "Extract the story text of this page in reading order and say who speaks each line. "
    "Label characters character1, character2, ... Use UNKNOWN if the speaker is unclear "
    "and NARRATION for narrator boxes. "
    "Exclude sound effects, titles, credits, page numbers, and text on signs or documents. "
    'Output a JSON list only: [{{"speaker":...,"text":...}}]. Use [] if the page has no story text.'
)

def context_block(prev_pages):
    if not prev_pages:
        return ""
    lines = [f"Page {i} result: " + json.dumps(p, ensure_ascii=False, separators=(",", ":"))
             for i, p in enumerate(prev_pages, 1)]
    return ("Earlier pages of this sequence (reuse the same labels for the same characters):\n"
            + "\n".join(lines) + "\n")

def target_text(page):
    return json.dumps(page, ensure_ascii=False, separators=(",", ":"))

def load_image(rel_path):
    img = Image.open(BASE / rel_path).convert("RGB")
    if PIXEL_BUDGET is None:
        return img
    w, h = img.size
    s = min(1.0, (PIXEL_BUDGET / (w * h)) ** 0.5)
    return img.resize((max(28, int(w * s) // 28 * 28), max(28, int(h * s) // 28 * 28)), Image.LANCZOS)

def make_messages(image, page_idx, prev_pages):
    text = context_block(prev_pages) + PAGE_PROMPT.format(n=page_idx + 1)
    return [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": text},
    ]}]

def build_examples(seqs):
    out = []
    for s in seqs:
        pages = labels[s["sequence_id"]]
        for k in range(3):
            out.append({"seq_id": s["sequence_id"], "page_idx": k,
                        "image_path": s["images"][k],
                        "prev": pages[:k], "target": pages[k]})
    return out

class MangaPageDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        e = self.examples[i]
        image = load_image(e["image_path"])
        prompt_text = processor.apply_chat_template(
            make_messages(image, e["page_idx"], e["prev"]),
            tokenize=False, add_generation_prompt=True)
        enc = processor(text=[prompt_text], images=[image], return_tensors="pt")
        tgt = processor.tokenizer(target_text(e["target"]) + "<|im_end|>",
                                  add_special_tokens=False, return_tensors="pt").input_ids

        input_ids = torch.cat([enc.input_ids, tgt], dim=1)
        labels_t  = torch.cat([torch.full_like(enc.input_ids, -100), tgt], dim=1)
        if "mm_token_type_ids" in enc:
            mm = enc["mm_token_type_ids"]
        else:
            mm = (enc.input_ids == model.config.image_token_id).long()
        mm = torch.cat([mm, torch.zeros_like(tgt)], dim=1)
        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": labels_t,
            "mm_token_type_ids": mm,
            "pixel_values": enc.pixel_values,
            "image_grid_thw": enc.image_grid_thw,
        }

In [10]:
# sanity check 

tok = processor.tokenizer
def n_tok(s): return len(tok(s, add_special_tokens=False).input_ids)

all_examples = build_examples(dev_seqs)
tgt_lens = [n_tok(target_text(e["target"])) for e in all_examples]
print(f"{len(all_examples)} page examples | answer tokens: "
      f"mean {sum(tgt_lens)/len(tgt_lens):.0f}, max {max(tgt_lens)}")
MAX_NEW_TOKENS = int(max(tgt_lens) * 1.25) + 50
print("MAX_NEW_TOKENS per page:", MAX_NEW_TOKENS)

ctx_lens = [n_tok(context_block(e["prev"])) for e in all_examples]
print("longest context text (tokens):", max(ctx_lens))

# check a page-3 example (longest kind of input)
ex = [e for e in build_examples(train_seqs) if e["page_idx"] == 2][0]
item = MangaPageDataset([ex])[0]
print({k: tuple(v.shape) for k, v in item.items()})
print("image tokens:", int((item["image_grid_thw"].prod(-1) // 4).sum()),
      "| supervised tokens:", int((item["labels"] != -100).sum()))

240 page examples | answer tokens: mean 141, max 411
MAX_NEW_TOKENS per page: 563
longest context text (tokens): 701
{'input_ids': (1, 1890), 'attention_mask': (1, 1890), 'labels': (1, 1890), 'mm_token_type_ids': (1, 1890), 'pixel_values': (6072, 1176), 'image_grid_thw': (1, 3)}
image tokens: 1518 | supervised tokens: 122


In [ ]:
# worst case memory test

def to_cuda(batch):
    out = {k: v.to("cuda") for k, v in batch.items()}
    out["pixel_values"] = out["pixel_values"].to(torch.bfloat16)
    return out

@torch.no_grad()
def eval_loss(loader):
    model.eval()
    tot, n = 0.0, 0
    for batch in loader:
        tot += model(**to_cuda(batch)).loss.item(); n += 1
    model.train()
    return tot / n

train_examples = build_examples(train_seqs)
val_examples   = build_examples(val_seqs)
val_loader = DataLoader(MangaPageDataset(val_examples), batch_size=1, shuffle=False,
                        collate_fn=lambda b: b[0])

if DEBUG_STEPS:
    hardest = sorted(train_examples,
                     key=lambda e: n_tok(context_block(e["prev"]) + target_text(e["target"])),
                     reverse=True)[:DEBUG_STEPS]
    loader = DataLoader(MangaPageDataset(hardest), batch_size=1, shuffle=False,
                        collate_fn=lambda b: b[0])
    epochs = 1
else:
    loader = DataLoader(MangaPageDataset(train_examples), batch_size=1, shuffle=True,
                        collate_fn=lambda b: b[0])
    epochs = EPOCHS

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
total_steps = math.ceil(len(loader) / GRAD_ACCUM) * epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, total_steps // 10), total_steps)

log_path = LOG_DIR / "finetune_v2_perpage.jsonl"
torch.cuda.reset_peak_memory_stats()
model.train()
start, micro, stop = time.time(), 0, False

for epoch in range(epochs):
    run_loss, run_n = 0.0, 0
    for step, batch in enumerate(loader):
        loss = model(**to_cuda(batch)).loss
        (loss / GRAD_ACCUM).backward()
        run_loss += loss.item(); run_n += 1; micro += 1

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        if micro % 16 == 0:
            print(f"ep {epoch+1} ex {step+1}/{len(loader)} loss {run_loss/run_n:.4f} | "
                  f"{time.time()-start:.0f}s | peak {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")

        if DEBUG_STEPS and micro >= DEBUG_STEPS:
            print(f"Debug run OK. {(time.time()-start)/micro:.1f} s/example, "
                  f"peak {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")
            stop = True
            break
    if stop:
        break

    v = eval_loss(val_loader)
    rec = {"epoch": epoch + 1, "train_loss": run_loss / run_n, "val_loss": v,
           "elapsed_s": time.time() - start}
    print(rec)
    with open(log_path, "a") as f:
        f.write(json.dumps(rec) + "\n")
    model.save_pretrained(ADAPTER_DIR / f"epoch_{epoch+1}")

Debug run OK. 35.6 s/example, peak 11.9 GB


In [6]:
# run on full training set.

MODEL_ID     = "Qwen/Qwen2-VL-7B-Instruct"
PIXEL_BUDGET = None              # None = full resolution
EPOCHS       = 3
LR           = 1e-4
GRAD_ACCUM   = 4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
DEBUG_STEPS  = None             #  None for full run

In [ ]:
# training loop

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def to_cuda(batch):
    out = {k: v.to("cuda") for k, v in batch.items()}
    out["pixel_values"] = out["pixel_values"].to(torch.bfloat16)
    return out

@torch.no_grad()
def eval_loss(loader):
    model.eval()
    tot, n = 0.0, 0
    for batch in loader:
        tot += model(**to_cuda(batch)).loss.item(); n += 1
    model.train()
    return tot / n

train_examples = build_examples(train_seqs)
val_examples   = build_examples(val_seqs)
val_loader = DataLoader(MangaPageDataset(val_examples), batch_size=1, shuffle=False,
                        collate_fn=lambda b: b[0])

if DEBUG_STEPS:
    hardest = sorted(train_examples,
                     key=lambda e: n_tok(context_block(e["prev"]) + target_text(e["target"])),
                     reverse=True)[:DEBUG_STEPS]
    loader = DataLoader(MangaPageDataset(hardest), batch_size=1, shuffle=False,
                        collate_fn=lambda b: b[0])
    epochs = 1
else:
    loader = DataLoader(MangaPageDataset(train_examples), batch_size=1, shuffle=True,
                        collate_fn=lambda b: b[0])
    epochs = EPOCHS

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
total_steps = math.ceil(len(loader) / GRAD_ACCUM) * epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, total_steps // 10), total_steps)

log_path = LOG_DIR / "finetune_v2_perpage.jsonl"
torch.cuda.reset_peak_memory_stats()
model.train()
start, micro, stop = time.time(), 0, False

for epoch in range(epochs):
    run_loss, run_n = 0.0, 0
    for step, batch in enumerate(loader):
        loss = model(**to_cuda(batch)).loss
        (loss / GRAD_ACCUM).backward()
        run_loss += loss.item(); run_n += 1; micro += 1

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        if micro % 16 == 0:
            print(f"ep {epoch+1} ex {step+1}/{len(loader)} loss {run_loss/run_n:.4f} | "
                  f"{time.time()-start:.0f}s | peak {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")

        if DEBUG_STEPS and micro >= DEBUG_STEPS:
            print(f"Debug run OK. {(time.time()-start)/micro:.1f} s/example, "
                  f"peak {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")
            stop = True
            break
    if stop:
        break

    v = eval_loss(val_loader)
    rec = {"epoch": epoch + 1, "train_loss": run_loss / run_n, "val_loss": v,
           "elapsed_s": time.time() - start}
    print(rec)
    with open(log_path, "a") as f:
        f.write(json.dumps(rec) + "\n")
    model.save_pretrained(ADAPTER_DIR / f"epoch_{epoch+1}")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


ep 1 ex 16/192 loss 0.3233 | 145s | peak 11.5 GB
ep 1 ex 32/192 loss 0.3135 | 270s | peak 11.5 GB
ep 1 ex 48/192 loss 0.2903 | 379s | peak 11.5 GB
ep 1 ex 64/192 loss 0.2720 | 493s | peak 12.2 GB
ep 1 ex 80/192 loss 0.2496 | 573s | peak 12.2 GB
ep 1 ex 96/192 loss 0.2634 | 651s | peak 12.2 GB
ep 1 ex 112/192 loss 0.2528 | 728s | peak 12.2 GB
ep 1 ex 128/192 loss 0.2415 | 807s | peak 12.2 GB
ep 1 ex 144/192 loss 0.2318 | 888s | peak 12.2 GB
ep 1 ex 160/192 loss 0.2253 | 963s | peak 12.2 GB
ep 1 ex 176/192 loss 0.2219 | 1038s | peak 12.2 GB
ep 1 ex 192/192 loss 0.2184 | 1121s | peak 12.2 GB
{'epoch': 1, 'train_loss': 0.2184227458225602, 'val_loss': 0.15093456767499447, 'elapsed_s': 1191.3987309932709}
ep 2 ex 16/192 loss 0.1018 | 1267s | peak 12.2 GB
ep 2 ex 32/192 loss 0.1000 | 1343s | peak 12.2 GB
ep 2 ex 48/192 loss 0.1058 | 1429s | peak 12.2 GB
ep 2 ex 64/192 loss 0.1064 | 1500s | peak 12.2 GB
ep 2 ex 80/192 loss 0.1045 | 1578s | peak 12.2 GB
ep 2 ex 96/192 loss 0.1065 | 1658s | peak

Loss pattern: train loss dropped consistently (0.218 → 0.099 → 0.061), val loss improved then plateaued

(0.151 → 0.112 → 0.113) — classic early stopping signal

Best epoch: 2 (lowest val loss 0.1118)

Epoch 3 shows mild overfitting: train loss still dropping but val loss slightly increased

In [11]:
# Clear current model from memory and reload base model
del model
torch.cuda.empty_cache()
import gc; gc.collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

In [12]:
# Load best epoch (epoch 2, lowest val loss)
from peft import PeftModel

# Load epoch 2 adapter
model = PeftModel.from_pretrained(model, ADAPTER_DIR / "epoch_2")
model.eval()
print("Epoch 2 adapter loaded!")
print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Epoch 2 adapter loaded!
GPU memory: 5.68 GB


In [13]:
# validation prediction on epoch-2 adapter

def parse_page(raw):
    out = []
    for m in re.finditer(r'\{[^{}]*\}', raw):
        try:
            o = json.loads(m.group())
        except json.JSONDecodeError:
            continue
        if isinstance(o, dict) and "speaker" in o and "text" in o:
            out.append({"speaker": str(o["speaker"]).strip(), "text": str(o["text"]).strip()})
    return out

@torch.no_grad()
def predict_page(rel_path, page_idx, prev_pages):
    image = load_image(rel_path)
    prompt_text = processor.apply_chat_template(
        make_messages(image, page_idx, prev_pages), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt").to("cuda")
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True)
    return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def run_split(seqs, out_dir):
    model.eval()
    with open(out_dir / "predictions.jsonl", "w") as pf, open(out_dir / "raw.jsonl", "w") as rf:
        for n, seq in enumerate(seqs, 1):
            pages, raws = [], []
            for k, rel in enumerate(seq["images"]):
                raw = predict_page(rel, k, pages)
                raws.append(raw); pages.append(parse_page(raw))
            pf.write(json.dumps({"sequence_id": seq["sequence_id"], "pages": pages}) + "\n")
            rf.write(json.dumps({"sequence_id": seq["sequence_id"], "raw": raws}) + "\n")
            pf.flush(); rf.flush()
            print(f"[{n}/{len(seqs)}] {seq['sequence_id']}")
    model.train()

run_split(val_seqs, OUT_DIR)

val_ids = {s["sequence_id"] for s in val_seqs}
with open(BASE / "development/labels.jsonl") as f, open(OUT_DIR / "val_references.jsonl", "w") as o:
    for line in f:
        if json.loads(line)["sequence_id"] in val_ids:
            o.write(line)
with open("../outputs/baseline/baseline_predictions.jsonl") as f, \
     open(OUT_DIR / "baseline_on_val_predictions.jsonl", "w") as o:
    for line in f:
        if json.loads(line)["sequence_id"] in val_ids:
            o.write(line)

[1/16] seq_ee880b9612205860
[2/16] seq_d8afceb0e8f9122d
[3/16] seq_96839890d69d7698
[4/16] seq_d8bc3c6ac1bb6017
[5/16] seq_0ef2ec294597f572
[6/16] seq_eaad741293da2ac1
[7/16] seq_fc4c2b84b6ec32ff
[8/16] seq_cd727990f22c777b
[9/16] seq_ec05f45d71c43f2c
[10/16] seq_de542b63e6b19bc2
[11/16] seq_84541a1f4b159018
[12/16] seq_4833e23eec54344b
[13/16] seq_c4448514b6de1ab9
[14/16] seq_5f407d0064bb4a43
[15/16] seq_07838fb26cd1be54
[16/16] seq_58858a2227cf3761


In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(model, ADAPTER_DIR / "epoch_3")
model.eval()
print("Epoch 3 adapter loaded!")
print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Epoch 3 adapter loaded!
GPU memory: 5.68 GB


In [ ]:
# validation prediction using epoch-3 adpater

def parse_page(raw):
    out = []
    for m in re.finditer(r'\{[^{}]*\}', raw):
        try:
            o = json.loads(m.group())
        except json.JSONDecodeError:
            continue
        if isinstance(o, dict) and "speaker" in o and "text" in o:
            out.append({"speaker": str(o["speaker"]).strip(), "text": str(o["text"]).strip()})
    return out

@torch.no_grad()
def predict_page(rel_path, page_idx, prev_pages):
    image = load_image(rel_path)
    prompt_text = processor.apply_chat_template(
        make_messages(image, page_idx, prev_pages), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt").to("cuda")
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True)
    return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def run_split(seqs, out_dir):
    model.eval()
    with open(out_dir / "predictions.jsonl", "w") as pf, open(out_dir / "raw.jsonl", "w") as rf:
        for n, seq in enumerate(seqs, 1):
            pages, raws = [], []
            for k, rel in enumerate(seq["images"]):
                raw = predict_page(rel, k, pages)
                raws.append(raw); pages.append(parse_page(raw))
            pf.write(json.dumps({"sequence_id": seq["sequence_id"], "pages": pages}) + "\n")
            rf.write(json.dumps({"sequence_id": seq["sequence_id"], "raw": raws}) + "\n")
            pf.flush(); rf.flush()
            print(f"[{n}/{len(seqs)}] {seq['sequence_id']}")
    model.train()

run_split(val_seqs, OUT_DIR)

val_ids = {s["sequence_id"] for s in val_seqs}
with open(BASE / "development/labels.jsonl") as f, open(OUT_DIR / "val_references.jsonl", "w") as o:
    for line in f:
        if json.loads(line)["sequence_id"] in val_ids:
            o.write(line)
with open("../outputs/baseline/baseline_predictions.jsonl") as f, \
     open(OUT_DIR / "baseline_on_val_predictions.jsonl", "w") as o:
    for line in f:
        if json.loads(line)["sequence_id"] in val_ids:
            o.write(line)

[1/16] seq_ee880b9612205860
[2/16] seq_d8afceb0e8f9122d
[3/16] seq_96839890d69d7698
[4/16] seq_d8bc3c6ac1bb6017
[5/16] seq_0ef2ec294597f572
[6/16] seq_eaad741293da2ac1
[7/16] seq_fc4c2b84b6ec32ff
[8/16] seq_cd727990f22c777b
[9/16] seq_ec05f45d71c43f2c
[10/16] seq_de542b63e6b19bc2
[11/16] seq_84541a1f4b159018
[12/16] seq_4833e23eec54344b
[13/16] seq_c4448514b6de1ab9
[14/16] seq_5f407d0064bb4a43
[15/16] seq_07838fb26cd1be54
[16/16] seq_58858a2227cf3761


Mistake : epoch-3 adapter predictions overwrite the epoch-2 adapter predictions.

Fix : first score epoch-3 predictions then re run epoch-2 again